In [ ]:
from IPython.display import clear_output

In [ ]:
import logging
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # FATAL
logging.getLogger('tensorflow').setLevel(logging.FATAL)

In [ ]:
import tensorflow as tf
import pandas as pd
import seaborn as sns

In [ ]:
DATA_PATH = "/kaggle/input/tuberculosis-tb-chest-xray-dataset/TB_Chest_Radiography_Database/"

In [ ]:
BATCH_SIZE       = 32
IMG_HEIGHT_WIDTH = 256
IMG_INPUT_SHAPE  = (IMG_HEIGHT_WIDTH, IMG_HEIGHT_WIDTH, 3)
MAX_EPOCHS       = 30

In [ ]:
DS_TRAIN = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_HEIGHT_WIDTH, IMG_HEIGHT_WIDTH),
    batch_size=BATCH_SIZE
)
DS_VALID = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_HEIGHT_WIDTH, IMG_HEIGHT_WIDTH),
    batch_size=BATCH_SIZE
)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
DS_TRAIN = DS_TRAIN.cache().prefetch(buffer_size=AUTOTUNE)
DS_VALID = DS_VALID.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
def get_model(tf_model):
    model = None
    if tf_model == "ResNet50":
        model = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor = tf.keras.Input(shape=IMG_INPUT_SHAPE))
    else:
        raise Exception('SUNXYZ', 'Model unknown')
    return_model = tf.keras.models.Sequential()
    return_model.add(model)
    return_model.add(tf.keras.layers.GlobalAveragePooling2D())
    return_model.add(tf.keras.layers.Dense(128, activation='relu'))
    return_model.add(tf.keras.layers.Dense(64, activation='relu'))
    return_model.add(tf.keras.layers.Dense(2, activation='softmax'))
    return return_model

In [ ]:
BASIC_MODEL = tf.keras.models.Sequential()
BASIC_MODEL.add(tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
BASIC_MODEL.add(tf.keras.layers.MaxPooling2D((2, 2)))
BASIC_MODEL.add(tf.keras.layers.Conv2D(64, (3, 3), activation='relu'))
BASIC_MODEL.add(tf.keras.layers.MaxPooling2D((2, 2)))
BASIC_MODEL.add(tf.keras.layers.Conv2D(64, (3, 3), activation='relu'))
BASIC_MODEL.add(tf.keras.layers.Flatten())
BASIC_MODEL.add(tf.keras.layers.Dense(256, activation='relu'))
BASIC_MODEL.add(tf.keras.layers.Dense(64, activation='relu'))
BASIC_MODEL.add(tf.keras.layers.Dense(2, activation='softmax'))

In [ ]:
RESNET50_MODEL    = get_model("ResNet50")

In [ ]:
RESNET50_MODEL.summary()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

def compile_and_fit(model):

    checkpoint = ModelCheckpoint(
        "OGbest_model.keras",          # file name
        monitor="val_loss",          # use validation loss (better)
        save_best_only=True,
        mode="min",
        verbose=1
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.compile(
        optimizer='adam',
        loss=tf.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    history = model.fit(
        DS_TRAIN,
        validation_data=DS_VALID,
        epochs=MAX_EPOCHS,
        callbacks=[checkpoint, early_stop]   # 🔥 ADD HERE
    )

    return history

In [ ]:
RESNET50_HISTORY = compile_and_fit(RESNET50_MODEL)

In [ ]:
df_RESNET50_HISTORY = pd.DataFrame(RESNET50_HISTORY.history)

In [ ]:
model.save("best_model.keras", include_optimizer=False)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(IMG_HEIGHT_WIDTH, IMG_HEIGHT_WIDTH))
    
    img_array = image.img_to_array(img)
    
    img_array = img_array / 255.0   # normalize (same as training)
    
    img_array = np.expand_dims(img_array, axis=0)  # shape → (1, 256, 256, 3)
    
    return img_array

In [ ]:
def predict_image(img_path):
    img = preprocess_image(img_path)
    
    prediction = model.predict(img)
    
    class_index = np.argmax(prediction)
    
    confidence = np.max(prediction)
    
    if class_index == 0:
        label = "Normal"
    else:
        label = "Tuberculosis"
    
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.4f}")

In [ ]:
predict_image("/kaggle/input/tuberculosis-tb-chest-xray-dataset/TB_Chest_Radiography_Database/Tuberculosis/Tuberculosis-10.png")